In [1]:
!pip install chromadb

In [2]:
!pip install langchain sentence_transformers

In [3]:
!pip install langchain-text-splitters

In [4]:
import os
import glob
from tqdm import tqdm
import hashlib
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Environment

In [5]:
import sys
import os

ENV_TYPE = 'auto' # 'colab' | 'kaggle'

if ENV_TYPE == 'auto':
    if 'google.colab' in sys.modules:
        ENV_TYPE = 'colab'
    elif os.path.exists('/kaggle'):
        ENV_TYPE = 'kaggle'
    else:
        ENV_TYPE = 'local'

print(f"🌍 Detected Environment: {ENV_TYPE.upper()}")

if ENV_TYPE == 'colab':
    from google.colab import drive
    from google.colab import userdata

    drive.mount('/content/drive')

    BASE_DIR = "/content/drive/MyDrive"
    DATASET_PATH = os.path.join(BASE_DIR, "dataset")
    QA_DATASET_PATH = os.path.join(BASE_DIR, "stackoverflow-pytorch.csv")
    VENV_PATH = "/content/ragas_venv"

elif ENV_TYPE == 'kaggle':
    from kaggle_secrets import UserSecretsClient

    DATASET_PATH = "/kaggle/input/dataset"
    QA_DATASET_PATH = "/kaggle/input/stackoverflow/stackoverflow-pytorch.csv"
    VENV_PATH = "/kaggle/working/ragas_venv"

else:
    DATASET_PATH = "./dataset"
    QA_DATASET_PATH = "./stackoverflow-pytorch.csv"
    VENV_PATH = "./ragas_venv"

🌍 Detected Environment: COLAB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
def get_secret(key_name):
    """Универсальная функция для получения секретов в Colab и Kaggle"""
    try:
        if ENV_TYPE == 'colab':
            return userdata.get(key_name)
        elif ENV_TYPE == 'kaggle':
            return UserSecretsClient().get_secret(key_name)
        else:
            return os.getenv(key_name) # Для локального запуска
    except Exception as e:
        print(f"Warning: Could not retrieve secret '{key_name}'. Error: {e}")
        return None

## Config

In [7]:
TOP_K = 5
CHUNK_SIZE=1000
CHUNK_OVERLAP=200

EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
EVAL_EMBEDDING_MODEL = "Snowflake/snowflake-arctic-embed-m"
RERANK_MODEL_NAME = "BAAI/bge-reranker-base"
LLM_MODEL_GENERATION = "gemini-2.0-flash"
LLM_MODEL_JUDGE = "gemini-2.0-flash"


## Retrieve

In [8]:
def fast_deduplicate_chunks(chunks):
    seen_hashes = set()
    unique_chunks = []

    for chunk in chunks:
        content_hash = hash(chunk.page_content.strip())

        if content_hash not in seen_hashes:
            seen_hashes.add(content_hash)
            unique_chunks.append(chunk)

    print(f"🚀 Быстрая дедупликация: {len(chunks)} → {len(unique_chunks)} чанков")
    return unique_chunks

def signature_based_deduplication(chunks):
    seen_signatures = set()
    unique_chunks = []

    for chunk in chunks:
        content_preview = chunk.page_content[:100].strip()
        content_length = len(chunk.page_content)
        signature = f"{content_preview}_{content_length}"

        if signature not in seen_signatures:
            seen_signatures.add(signature)
            unique_chunks.append(chunk)

    print(f"🚀 Дедупликация по сигнатуре: {len(chunks)} → {len(unique_chunks)} чанков")
    return unique_chunks

In [9]:
def load_deduplicated_documents(dataset_path):
    md_files = glob.glob(os.path.join(dataset_path, "**/*.md"), recursive=True)
    all_documents = []
    seen_files = set()

    print(f"Найдено {len(md_files)} .md файлов")

    for file_path in tqdm(md_files, desc="Загрузка с дедупликацией"):
        try:
            file_hash = hash_file(file_path)
            if file_hash in seen_files:
                continue
            seen_files.add(file_hash)

            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()

            document = Document(
                page_content=content,
                metadata={
                    "source": file_path,
                    "filename": os.path.basename(file_path),
                    "folder": os.path.dirname(os.path.relpath(file_path, dataset_path)),
                    "file_type": "markdown"
                }
            )
            all_documents.append(document)

        except Exception as e:
            print(f"Ошибка загрузки {file_path}: {e}")

    return all_documents

def hash_file(filepath):
    import hashlib
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        buf = f.read(8192)
        while buf:
            hasher.update(buf)
            buf = f.read(8192)
    return hasher.hexdigest()

In [10]:
documents = load_deduplicated_documents(DATASET_PATH)

chunker = RecursiveCharacterTextSplitter(
    chunk_size=1600,
    chunk_overlap=100,
    separators=[
        "\n# ", "\n## ", "\n### ", "\n\n", "\n", " ",
    ],
    length_function=len
)
chunks = chunker.split_documents(documents)
print(f"Создано {len(chunks)} чанков")

chunks = signature_based_deduplication(chunks)
model = SentenceTransformer(EMBEDDING_MODEL).to("cuda")

chroma_client = chromadb.PersistentClient(path="./chroma_fast")
collection = chroma_client.get_or_create_collection(
    name="docs_fast",
    metadata={"hnsw:space": "cosine"}
)

existing = collection.count()
if existing:
    print(f"Коллекция уже содержит {existing} чанков, пропускаю индексацию.")
else:
    batch_size = 200
    print("Создание эмбеддингов...")
    for i in tqdm(range(0, len(chunks), batch_size)):
        batch = chunks[i:i + batch_size]

        documents_batch = [chunk.page_content for chunk in batch]
        metadatas_batch = [chunk.metadata for chunk in batch]

        ids_batch = []
        for j, chunk in enumerate(batch):
            chunk_id = f"{i+j}_{hash(chunk.page_content[:50]) & 0xFFFFFF}"
            ids_batch.append(chunk_id)

        embeddings_batch = model.encode(documents_batch, normalize_embeddings=True).tolist()

        collection.add(
            embeddings=embeddings_batch,
            documents=documents_batch,
            metadatas=metadatas_batch,
            ids=ids_batch
        )

    print(f"Векторная БД готова. Чанков: {len(chunks)}")


Найдено 3104 .md файлов


Загрузка с дедупликацией: 100%|██████████| 3104/3104 [00:19<00:00, 161.84it/s]


Создано 8579 чанков
🚀 Дедупликация по сигнатуре: 8579 → 7715 чанков


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Коллекция уже содержит 7710 чанков, пропускаю индексацию.


In [11]:
print(f"Loading Reranker: {RERANK_MODEL_NAME}...")
reranker = CrossEncoder(RERANK_MODEL_NAME, device="cuda")
print("Reranker loaded.")

Loading Reranker: BAAI/bge-reranker-base...
Reranker loaded.


In [12]:
def chromadb_deduplication_search(collection, model, query, n_results=5):
    initial_results = collection.query(
        query_embeddings=model.encode([query]).tolist(),
        n_results=n_results * 3,  # Берем в 3 раза больше
        include=["documents", "metadatas", "distances"]
    )
    unique_results = fast_filter_duplicates(initial_results, n_results)

    return unique_results

def fast_filter_duplicates(results, n_results):
    if not results['documents']:
        return results

    seen_sources = set()
    unique_docs = []
    unique_metadatas = []
    unique_distances = []

    for doc, metadata, distance in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        source = metadata.get('source', 'unknown')
        preview = doc[:100]
        content_key = f"{source}:{preview}"

        if content_key not in seen_sources and len(unique_docs) < n_results:
            seen_sources.add(content_key)
            unique_docs.append(doc)
            unique_metadatas.append(metadata)
            unique_distances.append(distance)

    return {
        'documents': [unique_docs],
        'metadatas': [unique_metadatas],
        'distances': [unique_distances]
    }

In [13]:
def fast_contextual_search(collection, model, query, n_results=5):
    results = chromadb_deduplication_search(collection, model, query, n_results)

    formatted = []
    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    )):
        formatted.append({
            "rank": i + 1,
            "distance": distance,
            "source": metadata.get('source', 'N/A'),
            "preview": get_best_preview(doc, 150),
            "has_code": '```' in doc,
            "filename": os.path.basename(metadata.get('source', 'N/A'))
        })

    return formatted

def get_best_preview(content, length=150):
    code_match = re.search(r'```.*?```', content, re.DOTALL)
    if code_match:
        code_text = code_match.group(0)
        if len(code_text) > length:
            return code_text[:length] + "..."
        return code_text

    if len(content) > length:
        return content[:length] + "..."
    return content

In [14]:
import re

In [15]:
test_queries = [
    "How to check if CUDA is available",
    "What is PyTorch",
    "How to create a tensor",
    "neural network implementation"
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"🔍 ЗАПРОС: {query}")
    print(f"{'='*60}")
    results = fast_contextual_search(collection, model, query)

    for result in results:
        print(f"{result['rank']}. [dist: {result['distance']:.3f}] {result['filename']}")
        print(f"   {result['preview']}")
        print()


🔍 ЗАПРОС: How to check if CUDA is available
1. [dist: 0.179] cuda.md
   ```
x = torch.empty((8, 42), device=args.device)
net = Network().to(device=args.device)

```

2. [dist: 0.188] torch.cuda.is_available.md
   torch.cuda.is_available 

torch.cuda. is_available ( ) [source](ht...

3. [dist: 0.197] cuda.md
   torch.cuda 

This package adds support for CUDA tensor types. 

It implements the same...

4. [dist: 0.207] cuda.md
   ```

Note 

When assessing the availability of CUDA in a given environment ( [`is_available()`](../generated/torch.cuda.is_available.html#torch.cuda.i...

5. [dist: 0.211] torch.Tensor.is_cuda.md
   torch.Tensor.is_cuda 

Tensor. is_cuda 
:   Is `True`  if the Tensor is ...


🔍 ЗАПРОС: What is PyTorch
1. [dist: 0.216] cuda.md
   ### PyTorch API 

Warning 

This API is in beta and may change in future releases.

PyTorch exposes graphs via a raw [`torch.cuda.CUDAGraph`](../gener...

2. [dist: 0.237] torch.compiler_faq.md
   Starting in 2.1, `torch.compile`  unders

In [16]:
from dataclasses import dataclass
from typing import Sequence

In [17]:
@dataclass
class RetrievedChunk:
    id: int
    text: str
    source: str
    score: float

In [18]:
def retrieve_chunks_for_llm(collection, embed_model, query: str, n_results: int = 5) -> list[RetrievedChunk]:
    results = chromadb_deduplication_search(collection, embed_model, query, n_results)

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]

    chunks: list[RetrievedChunk] = []
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists)):
        chunks.append(
            RetrievedChunk(
                id=i + 1,
                text=doc,
                source=meta.get("source", "N/A"),
                score=1.0 - float(dist),
            )
        )
    return chunks

In [19]:
def retrieve_chunks(
    collection,
    embed_model,
    reranker_model,
    query: str,
    n_results: int = 5,
    fetch_k: int = 20
) -> list[RetrievedChunk]:
    results = chromadb_deduplication_search(collection, embed_model, query, n_results=fetch_k)

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    if not docs:
        return []

    pairs = [[query, doc] for doc in docs]
    scores = reranker_model.predict(pairs)
    ranked_candidates = list(zip(docs, metas, scores))
    ranked_candidates.sort(key=lambda x: x[2], reverse=True)

    chunks: list[RetrievedChunk] = []
    for i, (doc, meta, score) in enumerate(ranked_candidates[:n_results]):
        chunks.append(
            RetrievedChunk(
                id=i + 1,
                text=doc,
                source=meta.get("source", "N/A"),
                score=float(score),
            )
        )

    return chunks

In [20]:
import pandas as pd
import copy

test_query = "How to create a tensor?"
candidates_count = 20

retrieved_docs = retrieve_chunks_for_llm(collection, model, test_query, n_results=candidates_count)
original_rank_map = {chunk.text: i + 1 for i, chunk in enumerate(retrieved_docs)}

pairs = [[test_query, chunk.text] for chunk in retrieved_docs]
new_scores = reranker.predict(pairs)

reranked_docs = copy.deepcopy(retrieved_docs)

for chunk, score in zip(reranked_docs, new_scores):
    chunk.score = float(score)

reranked_docs = sorted(reranked_docs, key=lambda x: x.score, reverse=True)


data = []
for new_rank, chunk in enumerate(reranked_docs):
    original_rank = original_rank_map.get(chunk.text, -1)
    position_change = original_rank - (new_rank + 1)

    data.append({
        "New Rank": new_rank + 1,
        "Original Rank": original_rank,
        "Change": position_change,
        "Reranker Score": round(chunk.score, 4),
        "Source": chunk.source,
        "Text Snippet": chunk.text[:100].replace('\n', ' ') + "..."
    })

reranking_results = pd.DataFrame(data)

print(f"Query: {test_query}\n")
print(reranking_results.head(10).to_markdown(index=False))

filename = "reranking_analysis.csv"
reranking_results.to_csv(filename, index=False)
print(f"\nРезультаты сохранены в файл: {filename}")

Query: How to create a tensor?

|   New Rank |   Original Rank |   Change |   Reranker Score | Source                                                  | Text Snippet                                                                                            |
|-----------:|----------------:|---------:|-----------------:|:--------------------------------------------------------|:--------------------------------------------------------------------------------------------------------|
|          1 |              19 |       18 |           0.9048 | drive/MyDrive/dataset/data/cuda.md                      | ``` cuda = torch.device('cuda') x_cpu = torch.empty(2) x_gpu = torch.empty(2, device=cuda) x_cpu_lon... |
|          2 |               2 |        0 |           0.8365 | drive/MyDrive/dataset/sparse/torch.sparse_coo_tensor.md | # Create an empty sparse tensor with the following invariants: #   1. sparse_dim + dense_dim = len(S... |
|          3 |              16 |       13 |           0.7946

## Generate

In [21]:
from google import genai

In [22]:
SYSTEM_INSTRUCTIONS = """You are an assistant that answers questions about PyTorch 2.x and its ecosystem.
You must only use the context snippets below (PyTorch docs + curated StackOverflow answers).
If the context is not enough to answer, say you don't know and suggest where to look in the official docs.
Never invent APIs, arguments, or behavior that are not supported by the context."""

ANSWER_INSTRUCTIONS = """Answer format:
1) First, give a concise direct answer in 2–4 sentences.
2) Then provide a bullet list with details and short code examples if helpful.
3) Each bullet with factual claims must end with a citation in the form [§N], where N is the context id.
4) If multiple snippets support the same point, you can use [§1, §3].
5) After the bullets, add a small 'Where to read more' list with file paths or URLs."""


In [23]:
def render_context(chunks: Sequence[RetrievedChunk], max_chars: int = 14000) -> str:
    parts: list[str] = []
    total_len = 0

    for ch in chunks:
        header = f"[{ch.id}] {ch.source} (score={ch.score:.4f})"
        body = ch.text.strip()
        block = header + "\n" + body + "\n"
        if total_len + len(block) > max_chars:
            break
        parts.append(block)
        total_len += len(block)

    return "\n\n".join(parts)


In [24]:
def build_rag_prompt(question: str, chunks: Sequence[RetrievedChunk]) -> str:
    context_block = render_context(chunks)
    joined_sources = "\n".join(f"[§{c.id}] {c.source}" for c in chunks)

    prompt = f"""{SYSTEM_INSTRUCTIONS}

{ANSWER_INSTRUCTIONS}

User question:
{question}

Context snippets:
{context_block}

Remember:
- Base the answer only on the context snippets above.
- If the answer is not in the context, say that you don't know and suggest checking the relevant section in the PyTorch docs.
- Use the citation format [§N] that corresponds to the snippet ids.

List of snippet ids and sources:
{joined_sources}
"""
    return prompt


In [25]:
class GeminiGenerator:
    def __init__(
        self,
        model_name: str = LLM_MODEL_GENERATION,
        api_key: str | None = None,
        temperature: float = 0.1,
        max_output_tokens: int = 1024,
    ) -> None:
        if api_key is None:
            self.client = genai.Client()
        else:
            self.client = genai.Client(api_key=api_key)
        self.model_name = model_name
        self.temperature = temperature
        self.max_output_tokens = max_output_tokens

    def generate_answer(
        self,
        question: str,
        chunks: Sequence[RetrievedChunk],
    ) -> str:
        prompt = build_rag_prompt(question, chunks)

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt,
            config={
                "temperature": self.temperature,
                "max_output_tokens": self.max_output_tokens,
            },
        )

        return response.text

    def generate_pure_answer(
        self,
        question: str,
    ) -> str:
        response = self.client.models.generate_content(
            model=self.model_name,
            contents=question,
            config={
                "temperature": self.temperature,
                "max_output_tokens": self.max_output_tokens,
            },
        )

        return response.text



In [26]:
api_key = get_secret("GOOGLE_API_KEY")

if not api_key:
    api_key = "*"
    print("⚠️ API Key not found!")

gemini = GeminiGenerator(api_key=api_key, model_name=LLM_MODEL_GENERATION)

In [27]:
def rag_answer(query: str, n_results: int = 5) -> str:
    chunks = retrieve_chunks(
        collection=collection,
        embed_model=model,
        reranker_model=reranker,
        query=query,
        n_results=n_results,
        fetch_k=n_results * 4
    )

    if not chunks:
        return "I couldn't retrieve any relevant context for this question from the knowledge base."

    return gemini.generate_answer(query, chunks)

def pure_gemini_answer(query: str) -> str:
    return gemini.generate_pure_answer(query)

In [28]:
test_query = "In PyTorch 2.x, how can I check if CUDA is available without initializing the CUDA driver, to avoid issues with forked processes?"

print(f"\n{'='*60}")
print(f"🔍 ЗАПРОС: {test_query}")
print(f"{'='*60}")

answer = rag_answer(test_query, n_results=5)
print(answer)

print(f"\n{'-'*60}")

gemini_answer = pure_gemini_answer(test_query)
print(gemini_answer)


🔍 ЗАПРОС: In PyTorch 2.x, how can I check if CUDA is available without initializing the CUDA driver, to avoid issues with forked processes?
Yes, in PyTorch 2.x, you can check if CUDA is available without initializing the CUDA driver by setting the environment variable `PYTORCH_NVML_BASED_CUDA_CHECK=1` before importing PyTorch or calling `torch.cuda.is_available()` [§2, §3]. This directs `torch.cuda.is_available()` to attempt an NVML-based assessment instead of the CUDA Runtime API, which initializes the CUDA driver [§2].

Here's how to do it:

*   **Set the environment variable:** Before running your Python script, set `PYTORCH_NVML_BASED_CUDA_CHECK=1` in your shell. For example, in Linux/macOS: `export PYTORCH_NVML_BASED_CUDA_CHECK=1` [§2].
*   **Check CUDA availability:** Use `torch.cuda.is_available()` to check if CUDA is available.  When `PYTORCH_NVML_BASED_CUDA_CHECK=1` is set, `is_available()` will use NVML to check CUDA availability, avoiding the CUDA driver initialization [§2,

## Metrics

### Lexical

In [29]:
import pandas as pd

df = pd.read_csv(QA_DATASET_PATH)

df = df.rename(columns={
    "question_body": "question",
    "answer_body": "answer",
})
df = df[["question", "answer", "answer_score"]].dropna()


In [30]:
df = df[df["answer_score"] > 0].reset_index(drop=True)
eval_df = df.sample(100, random_state=42).reset_index(drop=True)

In [31]:
def _normalize(text: str) -> list[str]:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9_]+", " ", text)
    return [t for t in text.split() if t]

def _token_overlap(pred: str, gold: str) -> tuple[int, int, int]:
    p_tokens = _normalize(pred)
    g_tokens = _normalize(gold)
    if not p_tokens and not g_tokens:
        return 0, 0, 0
    common = {}
    for t in p_tokens:
        common[t] = common.get(t, 0) + 1
    overlap = 0
    for t in g_tokens:
        if common.get(t, 0) > 0:
            overlap += 1
            common[t] -= 1
    return overlap, len(p_tokens), len(g_tokens)

def squad_precision(pred: str, gold: str) -> float:
    overlap, p_len, g_len = _token_overlap(pred, gold)
    if p_len == 0 and g_len == 0:
        return 1.0
    if p_len == 0:
        return 0.0
    return overlap / p_len

def squad_recall(pred: str, gold: str) -> float:
    overlap, p_len, g_len = _token_overlap(pred, gold)
    if p_len == 0 and g_len == 0:
        return 1.0
    if g_len == 0:
        return 0.0
    return overlap / g_len

def squad_f1(pred: str, gold: str) -> float:
    p = squad_precision(pred, gold)
    r = squad_recall(pred, gold)
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)


### Semantic

In [32]:
import numpy as np

embed_model = SentenceTransformer(EVAL_EMBEDDING_MODEL)

def cosine(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

def embedding_sim(pred: str, gold: str) -> float:
    vecs = embed_model.encode([pred, gold])
    return cosine(vecs[0], vecs[1])


#### Evaluation

In [33]:
def evaluate_with_ground_truth(eval_df: pd.DataFrame, rag_fn, pure_fn, max_samples: int | None = None):
    if max_samples is not None and max_samples < len(eval_df):
        data = eval_df.sample(max_samples, random_state=42).reset_index(drop=True)
    else:
        data = eval_df.reset_index(drop=True)
    records = []

    for row in tqdm(data.itertuples(index=False), total=len(data)):
        q = row.question
        gold = row.answer

        rag_answer = rag_fn(q)
        pure_answer = pure_fn(q)

        rag_precision = squad_precision(rag_answer, gold)
        rag_recall = squad_recall(rag_answer, gold)
        rag_f1 = squad_f1(rag_answer, gold)

        pure_precision = squad_precision(pure_answer, gold)
        pure_recall = squad_recall(pure_answer, gold)
        pure_f1 = squad_f1(pure_answer, gold)

        records.append({
            "question": q,
            "gold": gold,
            "rag_answer": rag_answer,
            "pure_answer": pure_answer,
            "rag_precision": rag_precision,
            "rag_recall": rag_recall,
            "rag_f1": rag_f1,
            "pure_precision": pure_precision,
            "pure_recall": pure_recall,
            "pure_f1": pure_f1,
            "answer_similarity": embedding_sim(rag_answer, pure_answer),
        })

    results_df = pd.DataFrame(records)

    summary = {
        "rag_mean_f1": results_df["rag_f1"].mean(),
        "pure_mean_f1": results_df["pure_f1"].mean(),
        "answer_similarity": results_df["answer_similarity"].mean(),
        "rag_better_count": (results_df["rag_f1"] > results_df["pure_f1"]).sum(),
        "pure_better_count": (results_df["pure_f1"] > results_df["rag_f1"]).sum(),
    }

    return results_df, summary

In [35]:
ground_results, ground_summary = evaluate_with_ground_truth(eval_df, rag_answer, pure_gemini_answer, max_samples=100) # here

print("GROUND TRUTH (RAG vs PURE):", ground_summary)

100%|██████████| 100/100 [18:32<00:00, 11.12s/it]

GROUND TRUTH (RAG vs PURE): {'rag_mean_f1': np.float64(0.2209047908341951), 'pure_mean_f1': np.float64(0.20927182190936924), 'answer_similarity': np.float64(0.8506020039319993), 'rag_better_count': np.int64(64), 'pure_better_count': np.int64(36)}


### LLM-as-a-judge

In [46]:
import sys
import subprocess
import json
import os
import shutil
from tqdm.notebook import tqdm

def setup_ragas_env():
    venv_path = VENV_PATH
    python_exec = f"{venv_path}/bin/python"

    clean_env = os.environ.copy()
    clean_env.pop("PYTHONPATH", None)
    clean_env.pop("PYTHONHOME", None)

    if os.path.exists(venv_path):
        print(f"🔄 Удаляем старое окружение {venv_path}...")
        shutil.rmtree(venv_path)

    print("⚙️ Создаем новое изолированное окружение (virtualenv)...")

    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", venv_path], check=True)

    print("📦 Установка зависимостей в venv (в тихом режиме)...")

    subprocess.run([
        python_exec, "-m", "pip", "install", "-q", "--no-cache-dir",
        "pandas", "wrapt", "ragas", "langchain-google-genai", "datasets"
    ], check=True, env=clean_env)

    return python_exec

ragas_script_content = """
import os
import json
import sys

sys.path.append(os.getcwd())

import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

input_file = sys.argv[1]
output_file = sys.argv[2]
api_key = sys.argv[3]
judge_model_name = sys.argv[4]

os.environ["GOOGLE_API_KEY"] = api_key

try:
    with open(input_file, 'r') as f:
        data = json.load(f)

    dataset = Dataset.from_list(data)

    llm = ChatGoogleGenerativeAI(model=judge_model_name, temperature=0)
    embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

    print(">>> Ragas: Start Evaluation...", flush=True)
    results = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_recall],
        llm=llm,
        embeddings=embeddings
    )

    with open(output_file, 'w') as f:
        json.dump(results.to_pandas().to_dict(orient='records'), f)
    print(">>> Ragas: Done.", flush=True)

except Exception as e:
    print(f"CRITICAL ERROR INSIDE RAGAS: {e}", flush=True)
    sys.exit(1)
"""

def run_ragas_in_kaggle(df_prepared, api_key, judge_model_name):
    with open("run_eval.py", "w") as f:
        f.write(ragas_script_content)

    records = df_prepared.to_dict(orient='records')
    with open("temp_input.json", "w") as f:
        json.dump(records, f)

    try:
        python_exec = setup_ragas_env()
    except Exception as e:
        print(f"❌ Ошибка настройки окружения: {e}")
        return pd.DataFrame()

    env = os.environ.copy()
    env.pop("PYTHONPATH", None)
    env.pop("PYTHONHOME", None)

    print("⏳ Запуск подпроцесса Ragas...")
    try:
        result = subprocess.run(
            [python_exec, "run_eval.py", "temp_input.json", "temp_output.json", api_key, judge_model_name],
            capture_output=True,
            text=True,
            env=env
        )

        print(result.stdout)
        if result.stderr and "Error" in result.stderr and "sitecustomize" not in result.stderr:
             print("STDERR:", result.stderr)

        if os.path.exists("temp_output.json"):
            with open("temp_output.json", "r") as f:
                return pd.DataFrame(json.load(f))
        else:
            print("❌ Файл результата не создан. Скрипт упал.")
            return pd.DataFrame()

    except Exception as e:
        print(f"❌ Ошибка запуска subprocess: {e}")
        return pd.DataFrame()


print("🚀 Генерация ответов RAG...")

prepared_data = []

subset = eval_df.sample(100, random_state=42).reset_index(drop=True) # here

for i, row in tqdm(subset.iterrows(), total=len(subset)):
    q = row.question

    # Retrieve
    try:
        chunks = retrieve_chunks(
          collection=collection,
          embed_model=model,
          reranker_model=reranker,
          query=q,
          n_results=TOP_K,
          fetch_k=TOP_K * 4
      )
        contexts = [c.text for c in chunks]
    except Exception as e:
        print(f"Warning: Ошибка поиска для вопроса {i}: {e}")
        chunks = []
        contexts = []

    # Generate
    try:
        if chunks:
            ans = gemini.generate_answer(q, chunks)
        else:
            ans = "I couldn't find any relevant information to answer your question."
    except Exception as e:
        print(f"Warning: Ошибка генерации для вопроса {i}: {e}")
        ans = "Error generating answer."

    prepared_data.append({
        "question": q,
        "answer": ans,
        "contexts": contexts,
        "ground_truth": row.answer
    })

df_for_ragas = pd.DataFrame(prepared_data)

print(f"Данные готовы. Запускаем оценку на {len(df_for_ragas)} примерах...")

# Ragas Judge
rag_results = run_ragas_in_kaggle(df_for_ragas, api_key, judge_model_name=LLM_MODEL_JUDGE)

print("\n=== РЕЗУЛЬТАТЫ ===")
display(rag_results)

if not rag_results.empty:
    faithfulness = rag_results['faithfulness'].mean()
    answer_relevancy = rag_results['answer_relevancy'].mean()
    context_recall = rag_results['context_recall'].mean()

    print(f"Average Faithfulness: {faithfulness:.4f}")
    print(f"Average Answer Relevancy: {answer_relevancy:.4f}")
    print(f"Average Context Recall: {context_recall:.4f}")

    rag_score = (0.4 * faithfulness) + (0.4 * answer_relevancy) + (0.2 * context_recall)

    print(f"\n🏆 FINAL RAG SCORE: {rag_score:.4f} / 1.00")

🚀 Генерация ответов RAG...


  0%|          | 0/100 [00:00<?, ?it/s]

Данные готовы. Запускаем оценку на 100 примерах...
🔄 Удаляем старое окружение /content/ragas_venv...
⚙️ Создаем новое изолированное окружение (virtualenv)...
📦 Установка зависимостей в venv (в тихом режиме)...
⏳ Запуск подпроцесса Ragas...
>>> Ragas: Start Evaluation...
>>> Ragas: Done.


=== РЕЗУЛЬТАТЫ ===


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_recall
0,"<p>In A2C, the actor and critic algorithm, the...",[For further details regarding the algorithm w...,"Unfortunately, the provided context snippets d...",<p>I'm not quite clear what you mean to update...,0.166667,0.000000,0.000000
1,<p>I am looking to use the nn package for the ...,[PyTorch 2.0 NNModule Support \n==============...,"I am sorry, but the provided context snippets ...",<p>I have found a solution. The proxy blocks g...,0.400000,0.000000,0.000000
2,"<p>Please excuse the novice question, but is <...",[```\nFirst instance: 4\nSecond instance: tens...,"In PyTorch, a `Module` is a class that serves ...",<p>It's a simple container.</p>\n\n<p>From the...,1.000000,0.709544,0.066667
3,<p>Is there a <strong>simple</strong> way to z...,[Parameters\n: * **input** ( [*Tensor*](../t...,"Yes, you can zero the diagonal of a PyTorch te...",<p>I believe the simplest would be to use <a h...,1.000000,0.831678,1.000000
4,"<p>BTW, I'm happily running AlexNet on Torch; ...",[tlparse / TORCH_TRACE \n=====================...,It appears you're encountering network connect...,"<p>The rockspec code has a built-in, tacit req...",0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...
95,"<p><a href=""http://jsfiddle.net/KrA3D/"" rel=""n...",[### Combined or separate [`forward()`](../gen...,"I am sorry, but I cannot answer your question ...",<p>You must position the element (relatively o...,0.857143,0.000000,0.000000
96,<p>I'm a new one for the field of deep learnin...,[# lower to target backend\n\n```\n\nPlease fo...,"Yes, the embedding in `nn.Embedding` can be ch...",<p>One can either learn embeddings during the ...,0.500000,0.792963,0.000000
97,<p>I have a use case where I do forward for ea...,"[# warmup\n# In a real setting, use a few batc...","Based on your description, the autograd graph ...",<p>Let's start with a general discussion of ho...,0.125000,0.717266,0.307692
98,<p>I know about two ways to exclude elements o...,"[x = torch.tensor([1., 1.], requires_grad=True...","Yes, there is a difference between using `torc...",<p><code>tensor.detach()</code> creates a tens...,0.444444,0.819259,0.888889


Average Faithfulness: 0.5005
Average Answer Relevancy: 0.5047
Average Context Recall: 0.1677

🏆 FINAL RAG SCORE: 0.4357 / 1.00


## Save metrics

In [48]:
!pip install wandb

In [49]:
def generate_run_name(config):
    """
    Генерирует читаемое название эксперимента на основе конфига.
    Пример: Gem2.0_bge-base_sz1000_ov200_k5
    """
    gen_model = config.get("gen_model", "").split("/")[-1]
    gen_model = gen_model.replace("gemini-", "Gem").replace("-flash", "Fl").replace("-pro", "Pro")

    embed_model = config.get("embed_model", "").split("/")[-1]
    embed_model = embed_model.replace("-en-v1.5", "").replace("snowflake-", "Snow")

    sz = config.get("chunk_size", 0)
    ov = config.get("chunk_overlap", 0)
    k = config.get("top_k", 0)

    run_name = f"{gen_model}_{embed_model}_sz{sz}_ov{ov}_k{k}"

    return run_name

In [50]:
import wandb

wandb_key = get_secret("wandb_api_key")

if wandb_key:
    wandb.login(key=wandb_key)
else:
    print("⚠️ WandB API Key not found. Logging disabled.")

wandb.login(key=wandb_key)

experiment_config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "gen_model": LLM_MODEL_GENERATION,
    "judge_model": LLM_MODEL_JUDGE,
    "embed_model": EMBEDDING_MODEL,
    "top_k": TOP_K
}

run_name = generate_run_name(experiment_config)
print(f"🚀 Запуск эксперимента: {run_name}")

run = wandb.init(
    project="pytorch-rag-experiments",
    name=run_name,
    config=experiment_config
)

wandb.log({
    "global/rag_score": rag_score,
    "global/faithfulness":  faithfulness,
    "global/answer_relevancy":  answer_relevancy,
    "global/context_recall":  context_recall,
    "ground/rag_mean_f1": ground_summary["rag_mean_f1"],
    "ground/pure_mean_f1": ground_summary["pure_mean_f1"],
    "ground/answer_similarity": ground_summary["answer_similarity"],
    "ground/rag_better_count": ground_summary["rag_better_count"],
})

results_table = wandb.Table(dataframe=rag_results)
wandb.log({"rag_results_table": results_table})

wandb_table = wandb.Table(dataframe=reranking_results)
wandb.log({"reranking/analysis_table": wandb_table})

bad_cases = rag_results[rag_results['faithfulness'] < 0.5]
if not bad_cases.empty:
    wandb.log({"hallucinations": wandb.Table(dataframe=bad_cases)})

run.finish()

print("Данные отправлены в W&B")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.
wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


🚀 Запуск эксперимента: Gem2.0Fl_bge-base_sz1000_ov200_k5


global/answer_relevancy,▁
global/context_recall,▁
global/faithfulness,▁
global/rag_score,▁
ground/answer_similarity,▁
ground/pure_mean_f1,▁
ground/rag_better_count,▁
ground/rag_mean_f1,▁
global/answer_relevancy,0.50475
global/context_recall,0.16773
global/faithfulness,0.50054


Данные отправлены в W&B
